# 02 Linear Regression - Tesla Stock Trading

This is a complete, standalone pipeline for training a Linear Regression model to predict Tesla stock returns, including split adjustment and temporal cleanup.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

sns.set(style='whitegrid')

### 1. Data Cleaning & Feature Engineering
We handle the "Split Cliffs" and mixed date formats from the raw trading data.

In [ ]:
def prepare_tesla_data(path):
    df = pd.read_csv(path)
    
    # 1. Back-adjust prices for splits
    df['adj_close'] = df['Close']
    for i in range(len(df)-1, 0, -1):
        if df.loc[i, 'Split_Factor'] > 1.0:
            df.loc[:i-1, 'adj_close'] /= df.loc[i, 'Split_Factor']
            
    # 2. Target: Next Day % Return
    df['target'] = df['adj_close'].pct_change().shift(-1)
    
    # 3. Features
    features = ['RSI', 'Volatility_20d', 'Rel_Strength_SPY', 'SMA_50']
    df = df.dropna(subset=['target'] + features)
    return df, features

df, feature_cols = prepare_tesla_data('../_data/stock_tesla.csv')
X = df[feature_cols]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False) # Time-series order preserved

### 2. Model Training

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Linear Trading model trained.")

### 3. Evaluation & Visuals

In [ ]:
y_pred = model.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

plt.figure(figsize=(10, 5))
plt.plot(y_test.values, label='Actual Returns', alpha=0.6)
plt.plot(y_pred, label='Predicted Returns', alpha=0.8)
plt.title('Tesla Returns Prediction - Linear Baseline')
plt.legend()
plt.show()

### 4. Model Saving

In [ ]:
joblib.dump(model, '../_model/tesla_trading_linear_regression.joblib')
print("Model saved.")